## Test insertions

I have sampled local matches with different reads and error rates from `ref.fasta`. I have then inserted these local matches into `query/query.fasta` at random positions to create `query/with_insertions_{er}.fasta`. The file `ground_truth/{er}.tsv` shows the location of each local match in the `query/with_insertions_{er}.fasta`. 

This notebook tests if the ground truth file has the correct locations.  

In [3]:
from Bio import SeqIO
import pandas as pd
import random

Compare the original query file and the query with insertions. Check if matches from ground truth actually appear where they are supposed to.

In [6]:
rep = "0"
er = "0"

original_query_file = "../100kb/query/one_line_rep" + rep + ".fasta"
query_with_insertions_file = "../100kb/query/with_insertions_rep" + rep + "_e" + er + ".fasta"
# the ground truth file shows positions in the query with insertions
ground_truth_file = "../100kb/ground_truth/rep" + rep + "_e" + er + ".tsv"
local_matches_file = "../100kb/local_matches/rep" + rep + "_e" + er + ".fastq"

In [7]:
original_query = list(SeqIO.parse(original_query_file, "fasta"))
query_with_insertions = list(SeqIO.parse(query_with_insertions_file, "fasta"))
local_matches = list(SeqIO.parse(local_matches_file, "fastq"))
ground_truth = pd.read_csv(ground_truth_file, sep='\t')

Take a random local match from the local matches file. Find it's assigned position in the ground truth file and extract that location of the query with insertions file.

In [23]:
random.seed(42)
random_insertion = random.choice(local_matches)
random_insertion

SeqRecord(seq=Seq('GCCCCCTATAGTGTTTTACACACGGTTTAGCTCCAACTCTTGACACCAGTTACA...CTA', SingleLetterAlphabet()), id='40', name='40', description="40 start_position=69268,length=122,errors=0,reference_id='1',reference_file='ref_rep0.fasta'", dbxrefs=[])

In [27]:
ground_truth.loc[ground_truth['id'] == int(random_insertion.name)]

,id,position,length,reference_position,reference_length,errors
46,40,97913,122,69268,122,0


In [29]:
position = ground_truth.loc[ground_truth['id'] == int(random_insertion.name)]["position"].values[0]
position

97913

In [30]:
length = ground_truth.loc[ground_truth['id'] == int(random_insertion.name)]["length"].values[0]
length

122

In [31]:
original_insertion = str(random_insertion.seq)
original_insertion

'GCCCCCTATAGTGTTTTACACACGGTTTAGCTCCAACTCTTGACACCAGTTACACCGCGTGTAATTCAATCTTCCTCGCAACGGCGCCAGCGACTTGACACGCGTAATCCGCGTATTGGCTA'

In [32]:
inserted_into_query = str(query_with_insertions[0].seq[position:position + length])
inserted_into_query

'GCCCCCTATAGTGTTTTACACACGGTTTAGCTCCAACTCTTGACACCAGTTACACCGCGTGTAATTCAATCTTCCTCGCAACGGCGCCAGCGACTTGACACGCGTAATCCGCGTATTGGCTA'

In [33]:
assert(original_insertion == inserted_into_query),"Something went wrong with the insertion"

### Check insertions

Does the simulated local match actually appear at the position that is specified in the ground truth file

In [36]:
def check_insertion_correctness():
    random_insertion = random.choice(local_matches)
    position = ground_truth.loc[ground_truth['id'] == int(random_insertion.name)]["position"].values[0]
    length = ground_truth.loc[ground_truth['id'] == int(random_insertion.name)]["length"].values[0]
    original_insertion = str(random_insertion.seq)
    inserted_into_query = str(query_with_insertions[0].seq[position:position + length])
    assert(original_insertion == inserted_into_query),"Something went wrong with the insertion"
    print("Insertion " + random_insertion.name + " found at position " + str(position))

In [37]:
for i in range(10):
    check_insertion_correctness()

Insertion 1 found at position 78425
Insertion 47 found at position 41633
Insertion 17 found at position 45485
Insertion 15 found at position 76502
Insertion 14 found at position 82268
Insertion 8 found at position 90795
Insertion 47 found at position 41633
Insertion 6 found at position 94094
Insertion 43 found at position 93317
Insertion 47 found at position 41633


### Check positions before first insertion

Is the leading sequence (before the first insertion) equal for the original query and the query with insertions?

In [38]:
ground_truth.head()

,id,position,length,reference_position,reference_length,errors
0,10,609,63,41015,63,0
1,25,2177,54,62912,54,0
2,42,2296,51,16128,51,0
3,35,4272,133,17344,133,0
4,13,7318,194,91941,194,0


In [39]:
first_insertion_position = ground_truth.iloc[0]['position']

str(original_query[0].seq[0:first_insertion_position]) == \
str(query_with_insertions[0].seq[0:first_insertion_position])

True

### Check a sample of positions in the middle

In [40]:
# get random positions from the ground truth list to verify them
ran_ind_list = random.sample(range(len(ground_truth['id'])), 10)
ran_ind_list.sort()

In [43]:
for ind in ran_ind_list:
    if (ind == 0):
        continue
    if (ind == len(ground_truth['id'])):
        continue
    # the ground truth positions represent indices in the query with insertions
    previous_insert_position = ground_truth.iloc[ind-1]['position']
    previous_insert_length = ground_truth.iloc[ind-1]['length']
    downstream_bias = sum(ground_truth.truncate(after=(ind-1))['length'])

    insert_position = ground_truth.iloc[ind]['position']
    insert_length = ground_truth.iloc[ind]['length']
    upstream_bias = sum(ground_truth.truncate(after=(ind))['length'])

    next_insert_position = ground_truth.iloc[ind+1]['position']

    original_query_downstream = str(original_query[0].seq[previous_insert_position - downstream_bias \
                                                          + previous_insert_length \
                                                          :insert_position - downstream_bias])

    insertion_downstream = str(query_with_insertions[0].seq[previous_insert_position + \
                                                            previous_insert_length:insert_position])

    if (original_query_downstream == insertion_downstream):
        print("Verified downstream sequence for insertion " + str(ground_truth.iloc[ind]['id']))
    else:
        print("Incorrect downstream " + str(ground_truth.iloc[ind]['id']))

    original_query_upstream = str(original_query[0].seq[insert_position - upstream_bias \
                                                        + insert_length \
                                                        :next_insert_position - upstream_bias])
    insertion_upstream = str(query_with_insertions[0].seq[insert_position + insert_length:next_insert_position])

    if (original_query_upstream == insertion_upstream):
        print("Verified upstream sequence for insertion " + str(ground_truth.iloc[ind]['id']))
    else:
        print("Incorrect upstream " + str(ground_truth.iloc[ind]['id']))
    

Verified downstream sequence for insertion 25
Verified upstream sequence for insertion 25
Verified downstream sequence for insertion 42
Verified upstream sequence for insertion 42
Verified downstream sequence for insertion 28
Verified upstream sequence for insertion 28
Verified downstream sequence for insertion 26
Verified upstream sequence for insertion 26
Verified downstream sequence for insertion 45
Verified upstream sequence for insertion 45
Verified downstream sequence for insertion 32
Verified upstream sequence for insertion 32
Verified downstream sequence for insertion 1
Verified upstream sequence for insertion 1
Verified downstream sequence for insertion 38
Verified upstream sequence for insertion 38
Verified downstream sequence for insertion 11
Verified upstream sequence for insertion 11
Verified downstream sequence for insertion 27
Verified upstream sequence for insertion 27


### Check positions after last insertion

Is the trailing sequence (after the last insertion) equal for the original query and the query with insertions?

In [44]:
ground_truth.tail()

,id,position,length,reference_position,reference_length,errors
46,40,97913,122,69268,122,0
47,49,101447,52,65512,52,0
48,9,103550,136,16363,136,0
49,27,103829,62,8200,62,0
50,18,105810,191,89013,191,0


In [45]:
last_insertion_position = ground_truth.iloc[-1]['position']
last_insertion_length = ground_truth.iloc[-1]['length']
bias = sum(ground_truth["length"]) - last_insertion_length

str(original_query[0].seq[last_insertion_position - bias:]) == \
str(query_with_insertions[0].seq[last_insertion_position + last_insertion_length:])

True